<a href="https://colab.research.google.com/github/karthikeyan06-cyber/Flipkart-project-/blob/main/Shopper_Spectrum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    -Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce



##### **Project Type**    - EDA
##### **Contribution**    - Individual

# **Project Summary**-Shopper Spectrum


**UK E-Commerce Dataset | 541,909 Transactions**

---

## Project Overview

Shopper Spectrum is an end-to-end machine learning project built on a UK-based online retail dataset spanning one year of transactions (Dec 2022 – Dec 2023). The project delivers two production-ready capabilities: behavioural customer segmentation using unsupervised learning, and a product recommendation engine using collaborative filtering — both served through an interactive Streamlit web application.

---

## Dataset & Preprocessing

The raw dataset contained **541,909 rows** across 8 columns (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country). Data quality issues identified and resolved:

- Dropped **135,080 rows (24.9%)** with missing CustomerID — cannot build customer profiles without an identifier
- Removed **9,288 cancelled invoices** (InvoiceNo prefixed 'C') to avoid inflating purchase counts
- Filtered negative Quantity and UnitPrice values caused by returns and data entry errors
- Parsed InvoiceDate to datetime; engineered `TotalAmount = Quantity × UnitPrice`

**Clean dataset: 397,884 rows (73.4% retained) | 4,338 unique customers | 3,877 unique products**

---

## RFM Feature Engineering

Each customer's purchase history was distilled into three metrics — **Recency** (days since last purchase), **Frequency** (distinct invoice count), and **Monetary** (total spend in £). Both Frequency and Monetary exhibited extreme right-skew (skewness of 12.1 and 19.3 respectively), caused by a small number of wholesale-level buyers. A `log1p` transformation was applied before `StandardScaler` normalisation to produce a symmetric, mean-zero feature space suitable for KMeans clustering.

---

## Customer Segmentation (KMeans)

KMeans clustering was evaluated for k = 2 to 10. The Elbow method and Silhouette Score (peak at k=4: **0.337**) both indicated k=4 as the optimal choice, balancing statistical quality with business interpretability. Cluster labels were assigned dynamically by ranking clusters on a composite RFM score:

| Segment | Avg Recency | Avg Frequency | Avg Monetary | Customers | Share |
|---|---|---|---|---|---|
|  High-Value | 12 days | 13.7 orders | £8,074 | 716 | 16.5% |
|  Regular | 71 days | 4.1 orders | £1,803 | 1,173 | 27.0% |
|  Occasional | 18 days | 2.1 orders | £552 | 837 | 19.3% |
|  At-Risk | 182 days | 1.3 orders | £343 | 1,612 | 37.2% |

---

## Recommendation System (Collaborative Filtering)

An item-based collaborative filtering system was built using a **4,338 × 3,260 Customer × Product pivot matrix** (products bought by fewer than 5 customers were excluded as noise). Cosine similarity was computed across all product pairs, producing a 3,260 × 3,260 similarity matrix. Given any product, the system returns the top-N most co-purchased items with no reliance on product metadata — purely from purchase behaviour. Validation confirmed strong results: *JUMBO BAG RED RETROSPOT* correctly surfaces *JUMBO BAG STRAWBERRY* as its nearest neighbour (similarity: **0.902**).

---

## Streamlit Application

A two-tab Streamlit app serves both models in real time. All five model artifacts are cached on startup using `@st.cache_resource` for instant inference.

- **Tab 1 — Product Recommendations:** dropdown of all 3,260 products with adjustable top-N slider; results shown with cosine similarity scores
- **Tab 2 — Customer Segmentation:** Recency, Frequency, Monetary inputs run through the exact `log1p → StandardScaler → KMeans` pipeline used during training

---

## Tech Stack

`Python` · `pandas` · `NumPy` · `scikit-learn` · `Matplotlib` · `Seaborn` · `joblib` · `Streamlit`

# **GitHub Link -**

# **Problem Statement**


The global e-commerce industry generates vast amounts of transaction data daily, offering valuable insights into customer purchasing behaviors. Analyzing this data is essential for identifying meaningful customer segments and recommending relevant products to enhance customer experience and drive business growth. This project aims to examine transaction data from an online retail business to uncover patterns in customer purchase behavior, segment customers based on Recency, Frequency, and Monetary (RFM) analysis, and develop a product recommendation system using collaborative filtering techniques.


## ***1. Know Your Data***

### 1.1 Import Libraries

In [90]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

### 1.2 Dataset Loading

In [91]:
# Load Dataset
df = pd.read_csv('online_retail.csv', encoding='latin1')

print(" Dataset loaded successfully!")


 Dataset loaded successfully!


### Dataset First View


In [92]:
# Dataset First Look

print("\n First 5 rows:")
print(df.head())


 First 5 rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2022-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2022-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2022-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2022-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2022-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [93]:
#  1.2 Basic Structure
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

   Rows    : 541,909
   Columns : 8


### 1.3 Dataset Information

In [94]:
print(df.info())
print("\n Column names and data types:")
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None

 Column names and data types:
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


### 1.4 Duplicate Values and Missing Values

In [95]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 5268


In [96]:
print("\n  Missing value counts:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_summary[missing_summary['Missing Count'] > 0])


  Missing value counts:
             Missing Count  Missing %
Description           1454       0.27
CustomerID          135080      24.93


###1.5 Visualizing the missing values

In [97]:
# Plot: Missing values
fig, ax = plt.subplots(1, 1, figsize=(8, 6)) # Create a single subplot
fig.suptitle('Step 1 — Dataset Overview', fontsize=15, fontweight='bold')

missing_cols = missing[missing > 0]
ax.bar(missing_cols.index, missing_cols.values, color='tomato')
ax.set_title('Missing Value Counts')
ax.set_ylabel('Missing Rows')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

In [98]:

print("\n Statistical summary (numeric columns):")
print(df.describe())



 Statistical summary (numeric columns):
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


### 1.6 Unique Value Counts

In [99]:

print("\n Unique value counts per column:")
for col in df.columns:
    print(f"   {col:<15}: {df[col].nunique():>6,} unique values")



 Unique value counts per column:
   InvoiceNo      : 25,900 unique values
   StockCode      :  4,070 unique values
   Description    :  4,223 unique values
   Quantity       :    722 unique values
   InvoiceDate    : 23,260 unique values
   UnitPrice      :  1,630 unique values
   CustomerID     :  4,372 unique values
   Country        :     38 unique values


###  1.7 Cancelled Invoices

In [100]:

cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"\n Cancelled invoices (InvoiceNo starts with 'C'): {len(cancelled):,}")
print(f"   That's {len(cancelled)/len(df)*100:.2f}% of all rows")



 Cancelled invoices (InvoiceNo starts with 'C'): 9,288
   That's 1.71% of all rows


###1.8 Data Range

In [101]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')
print(f"\n Date range:")
print(f"   Earliest : {df['InvoiceDate'].min()}")
print(f"   Latest   : {df['InvoiceDate'].max()}")
print(f"   Span     : ~1 year of transactions (Dec 2022 – Dec 2023)")


 Date range:
   Earliest : 2022-12-01 08:26:00
   Latest   : 2023-12-09 12:50:00
   Span     : ~1 year of transactions (Dec 2022 – Dec 2023)


###1.9 Country Distribution

In [102]:

print("\n Top 10 countries by transaction count:")
print(df['Country'].value_counts().head(10))


 Top 10 countries by transaction count:
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64


# **Visualisations**

In [103]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Step 1 — Dataset Overview', fontsize=15, fontweight='bold')

# Plot 1: Top 10 countries
top_countries = df['Country'].value_counts().head(10)
axes[0, 0].barh(top_countries.index[::-1], top_countries.values[::-1], color='steelblue')
axes[0, 0].set_title('Transactions by Country (Top 10)')
axes[0, 0].set_xlabel('Number of Transactions')

# Plot 2: Missing values
missing_cols = missing[missing > 0]
axes[0, 1].bar(missing_cols.index, missing_cols.values, color='tomato')
axes[0, 1].set_title('Missing Value Counts')
axes[0, 1].set_ylabel('Missing Rows')

# Plot 3: Quantity distribution (clipped to -100 to 100 for readability)
qty_clipped = df['Quantity'].clip(-100, 200)
axes[1, 0].hist(qty_clipped, bins=60, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Quantity Distribution (clipped -100 to 200)')
axes[1, 0].set_xlabel('Quantity')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
axes[1, 0].legend()

# Plot 4: UnitPrice distribution (log scale)
positive_prices = df[df['UnitPrice'] > 0]['UnitPrice']
axes[1, 1].hist(positive_prices, bins=80, color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('UnitPrice Distribution (positive values only)')
axes[1, 1].set_xlabel('Unit Price (£)')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('step1_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n Visualisation saved as step1_overview.png")



 Visualisation saved as step1_overview.png


# 2. Data Preprocessing & Cleaning

###  2.1  Load raw data

In [104]:
df = pd.read_csv('online_retail.csv', encoding='latin1')
print(f"Raw dataset : {len(df):,} rows")


Raw dataset : 541,909 rows


###  2.2  Track row counts at each cleaning step

In [105]:
cleaning_log = []
cleaning_log.append(("Raw data", len(df)))

###2.3 Drop rows with missing CustomerID

In [106]:
df = df.dropna(subset=['CustomerID'])
cleaning_log.append(("Drop null CustomerID", len(df)))
print(f"After drop null CustomerID : {len(df):,} rows")

After drop null CustomerID : 406,829 rows


 RFM analysis is customer-level. Without a CustomerID
      we have no idea about who made the purchase.It is useless for clustering
###  **135,080 rows** (~25%) have no CustomerID.

###  2.4   Remove cancelled invoices

In [107]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
cleaning_log.append(("Remove cancelled invoices", len(df)))
print(f"After remove cancellations : {len(df):,} rows")

After remove cancellations : 397,924 rows


Invoices starting with 'C' are cancelled invoices .Including them would reduce the  Frequency counts and undercount Monetary values.

###  2.5  Remove non-positive Quantity



In [108]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
cleaning_log.append(("Remove Quantity ≤ 0", len(df)))
cleaning_log.append(("Remove UnitPrice ≤ 0", len(df)))
print(f"After remove Quantity ≤ 0 : {len(df):,} rows")
print(f"After remove UnitPrice ≤ 0 : {len(df):,} rows")

After remove Quantity ≤ 0 : 397,884 rows
After remove UnitPrice ≤ 0 : 397,884 rows



* Quantity with **(-)ve values** will indicates returns, adjustments, or data errors. We only want actual purchases.

*    Prices with **(-)ve values** are **0** are errors or internal adjustments.
      A £0 price gives a £0 TotalAmount which distorts
      the Monetary metric.



### 2.7 Fix data types

In [109]:
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"\nCustomerID dtype : {df['CustomerID'].dtype}  (sample: {df['CustomerID'].iloc[0]})")
print(f"InvoiceDate dtype: {df['InvoiceDate'].dtype}")



CustomerID dtype : object  (sample: 17850)
InvoiceDate dtype: datetime64[ns]


### 2.8 Engineer TotalAmount

In [110]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

### 2.9  Final null check

In [111]:
print("\n Null check after cleaning:")
print(df.isnull().sum())


 Null check after cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalAmount    0
dtype: int64


###2.10 Data Preprocessing & Cleaning **(Summary)**

In [112]:
total_raw   = cleaning_log[0][1]
total_clean = len(df)
removed     = total_raw - total_clean
print("Clean dataset preview:")
print(df.head())
print()
print("Clean dataset dtypes:")
print(df.dtypes)
print()
print("TotalAmount stats:")
print(df['TotalAmount'].describe())

Clean dataset preview:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice CustomerID         Country  TotalAmount  
0 2022-12-01 08:26:00       2.55      17850  United Kingdom        15.30  
1 2022-12-01 08:26:00       3.39      17850  United Kingdom        20.34  
2 2022-12-01 08:26:00       2.75      17850  United Kingdom        22.00  
3 2022-12-01 08:26:00       3.39      17850  United Kingdom        20.34  
4 2022-12-01 08:26:00       3.39      17850  United Kingdom        20.34  

Clean dataset dtypes:
InvoiceNo              object
StockCode              object
Description

### 2.11 **Visualation**

In [113]:
# ── 2.11 Visualisation — Before / After waterfall ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Step 2 — Data Cleaning Overview', fontsize=14, fontweight='bold')

# LEFT: Waterfall chart showing rows removed at each step
labels   = [label for label, _ in cleaning_log]
counts   = [count for _, count in cleaning_log]
removed_each = [counts[i-1] - counts[i] for i in range(1, len(counts))]

colors_bar = ['#4C72B0'] + ['#DD8452'] * len(removed_each)
bar_heights = [counts[0]] + removed_each

ax = axes[0]
ax.bar(labels, counts, color=colors_bar, edgecolor='white', linewidth=0.5)
ax.set_title('Row count after each cleaning step')
ax.set_ylabel('Number of rows')
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
for i, v in enumerate(counts):
    ax.text(i, v + 3000, f'{v:,}', ha='center', fontsize=7.5, fontweight='bold')
raw_patch   = mpatches.Patch(color='#4C72B0', label='Kept rows')
drop_patch  = mpatches.Patch(color='#DD8452', label='Dropped rows')
ax.legend(handles=[raw_patch, drop_patch], fontsize=8)

# RIGHT: Pie chart — what % of raw data did we keep vs drop
kept    = total_clean
dropped = total_raw - total_clean
axes[1].pie(
    [kept, dropped],
    labels=[f'Kept\n{kept:,} ({kept/total_raw*100:.1f}%)',
            f'Dropped\n{dropped:,} ({dropped/total_raw*100:.1f}%)'],
    colors=['#4C72B0', '#DD8452'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Overall: Kept vs Dropped rows')

plt.tight_layout()
plt.savefig('step2_cleaning.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualisation saved as step2_cleaning.png")

/tmp/ipykernel_973/3910193752.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)


📊 Visualisation saved as step2_cleaning.png


### 2.12 Save clean dataframe for next steps

In [114]:
df.to_csv('online_retail_clean.csv', index=False)
print("💾 Clean data saved as online_retail_clean.csv")

💾 Clean data saved as online_retail_clean.csv


# 3.0 Exploratory Data Analysis (EDA)

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


###  Load the clean data from Step 2

In [116]:
df = pd.read_csv('online_retail_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype(str)

print(f"Working with {len(df):,} clean transactions across "
      f"{df['CustomerID'].nunique():,} unique customers")

Working with 397,884 clean transactions across 4,338 unique customers


### 3.1  PRODUCT-LEVEL ANALYSIS

In [117]:
top_products = df.groupby('Description')['Quantity'].sum().nlargest(10)

plt.figure(figsize=(10, 6))
top_products.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Best-Selling Products (by Quantity)', fontsize=13, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.tight_layout()
plt.savefig('eda_top_products.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2  GEOGRAPHIC ANALYSIS

In [118]:
top_countries = df['Country'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_countries.sort_values().plot(kind='barh', color='coral')
plt.title('Top 10 Countries by Transaction Count', fontsize=13, fontweight='bold')
plt.xlabel('Number of Transactions')
plt.tight_layout()
plt.savefig('eda_countries.png', dpi=150, bbox_inches='tight')
plt.show()

uk_share = (df['Country'] == 'United Kingdom').mean() * 100
print(f"\n🇬🇧 United Kingdom makes up {uk_share:.1f}% of all transactions")


🇬🇧 United Kingdom makes up 89.1% of all transactions


### 3.3  TIME-SERIES ANALYSIS





In [119]:
monthly_sales = df.set_index('InvoiceDate').resample('ME')['TotalAmount'].sum()

plt.figure(figsize=(12, 5))
monthly_sales.plot(kind='line', marker='o', color='seagreen', linewidth=2)
plt.title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
plt.ylabel('Total Revenue (£)')
plt.xlabel('Month')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('eda_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4  CUSTOMER-LEVEL ANALYSIS

In [120]:
customer_freq = df.groupby('CustomerID')['InvoiceNo'].nunique()

plt.figure(figsize=(10, 5))
plt.hist(customer_freq.clip(upper=30), bins=30, color='mediumpurple', edgecolor='white')
plt.title('Distribution of Purchase Frequency per Customer (clipped at 30)',
          fontsize=13, fontweight='bold')
plt.xlabel('Number of Distinct Invoices (Frequency)')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('eda_frequency_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📦 Frequency stats (orders per customer):")
print(customer_freq.describe())


📦 Frequency stats (orders per customer):
count    4338.000000
mean        4.272015
std         7.697998
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: InvoiceNo, dtype: float64


### 3.5  CUSTOMER-LEVEL ANALYSIS

In [121]:
customer_monetary = df.groupby('CustomerID')['TotalAmount'].sum()

plt.figure(figsize=(10, 5))
plt.hist(np.log1p(customer_monetary), bins=50, color='darkorange', edgecolor='white')
plt.title('Distribution of Total Spend per Customer (log scale)',
          fontsize=13, fontweight='bold')
plt.xlabel('log(1 + Total Spend in £)')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('eda_monetary_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💰 Monetary stats (total spend per customer):")
print(customer_monetary.describe())


💰 Monetary stats (total spend per customer):
count      4338.000000
mean       2054.266460
std        8989.230441
min           3.750000
25%         307.415000
50%         674.485000
75%        1661.740000
max      280206.020000
Name: TotalAmount, dtype: float64


### 3.6  CORRELATION CHECK

In [122]:
rfm_preview = pd.DataFrame({
    'Frequency': customer_freq,
    'Monetary': customer_monetary
})

plt.figure(figsize=(8, 6))
plt.scatter(rfm_preview['Frequency'], rfm_preview['Monetary'],
            alpha=0.4, color='teal', s=20)
plt.title('Frequency vs Monetary (per customer)', fontsize=13, fontweight='bold')
plt.xlabel('Frequency (number of orders)')
plt.ylabel('Monetary (total spend £)')
plt.yscale('log')
plt.tight_layout()
plt.savefig('eda_freq_vs_monetary.png', dpi=150, bbox_inches='tight')
plt.show()

correlation = rfm_preview['Frequency'].corr(rfm_preview['Monetary'])
print(f"\n Correlation between Frequency and Monetary: {correlation:.3f}")


 Correlation between Frequency and Monetary: 0.554


# 4.0 RFM Feature Engineering

In [123]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
df = pd.read_csv('online_retail_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype(str)

### 4.1  BUILD THE RFM TABLE

In [124]:
reference_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"Reference date for Recency calculation: {reference_date}")

Reference date for Recency calculation: 2023-12-10 12:50:00


 **RFM = Recency,Frequency, Monetary** — three numbers that
 summarise a customer's entire purchase history into a
 single row. This is the input to KMeans clustering.

In [125]:
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalAmount', 'sum')
).reset_index()

print(f"\nRFM table built for {len(rfm):,} unique customers")
print(rfm.head(10))



RFM table built for 4,338 unique customers
  CustomerID  Recency  Frequency  Monetary
0      12346      326          1  77183.60
1      12347        2          7   4310.00
2      12348       75          4   1797.24
3      12349       19          1   1757.55
4      12350      310          1    334.40
5      12352       36          8   2506.04
6      12353      204          1     89.00
7      12354      232          1   1079.40
8      12355      214          1    459.40
9      12356       23          3   2811.43



### Recency   → days since their LAST purchase. LOWER = better

### Frequency → number of DISTINCT invoices (orders), not line items. HIGHER = better (loyal/repeat customer)
### Monetary  → total amount spent (£) across all orders.             HIGHER = better (valuable customer)

### 4.2  CHECK THE DISTRIBUTION OF EACH METRIC

In [126]:
print("\n RFM descriptive statistics:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

print("\n Skewness (0 = symmetric, >1 = highly skewed):")
for col in ['Recency', 'Frequency', 'Monetary']:
    print(f"   {col:<10}: {rfm[col].skew():.2f}")


 RFM descriptive statistics:
           Recency    Frequency       Monetary
count  4338.000000  4338.000000    4338.000000
mean     92.536422     4.272015    2054.266460
std     100.014169     7.697998    8989.230441
min       1.000000     1.000000       3.750000
25%      18.000000     1.000000     307.415000
50%      51.000000     2.000000     674.485000
75%     142.000000     5.000000    1661.740000
max     374.000000   209.000000  280206.020000

 Skewness (0 = symmetric, >1 = highly skewed):
   Recency   : 1.25
   Frequency : 12.07
   Monetary  : 19.32


In [127]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'],
                            ['steelblue','seagreen','coral']):
    ax.hist(rfm[col], bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution (raw)')
fig.suptitle('Step 4.2 — Raw RFM Distributions (before transform)', fontweight='bold')
plt.tight_layout()
plt.savefig('rfm_raw_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.3  LOG TRANSFORM

In [128]:
rfm['Recency_log']   = np.log1p(rfm['Recency'])
rfm['Frequency_log'] = np.log1p(rfm['Frequency'])
rfm['Monetary_log']  = np.log1p(rfm['Monetary'])

print("\n Skewness AFTER log transform:")
for col in ['Recency_log', 'Frequency_log', 'Monetary_log']:
    print(f"   {col:<15}: {rfm[col].skew():.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['Recency_log','Frequency_log','Monetary_log'],
                            ['steelblue','seagreen','coral']):
    ax.hist(rfm[col], bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution (log-transformed)')
fig.suptitle('Step 4.3 — RFM Distributions AFTER log transform', fontweight='bold')
plt.tight_layout()
plt.savefig('rfm_log_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


 Skewness AFTER log transform:
   Recency_log    : -0.38
   Frequency_log  : 1.21
   Monetary_log   : 0.39


### 4.4  SCALE THE FEATURES

In [129]:
scaler = StandardScaler()
rfm_scaled_array = scaler.fit_transform(rfm[['Recency_log', 'Frequency_log', 'Monetary_log']])

rfm_scaled = pd.DataFrame(
    rfm_scaled_array,
    columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'],
    index=rfm.index)

In [130]:
# Combine back with CustomerID for reference
rfm_final = pd.concat([rfm[['CustomerID']], rfm_scaled], axis=1)

print("\n✅ Scaled RFM features (mean ≈ 0, std ≈ 1):")
print(rfm_scaled.describe())
print()
print(rfm_final.head())


✅ Scaled RFM features (mean ≈ 0, std ≈ 1):
       Recency_scaled  Frequency_scaled  Monetary_scaled
count    4.338000e+03      4.338000e+03     4.338000e+03
mean    -9.172520e-17     -7.206980e-17     2.948310e-16
std      1.000115e+00      1.000115e+00     1.000115e+00
min     -2.341296e+00     -9.552143e-01    -4.004574e+00
25%     -6.613615e-01     -9.552143e-01    -6.856676e-01
50%      8.992557e-02     -3.615828e-01    -6.218718e-02
75%      8.447915e-01      6.532370e-01     6.541861e-01
max      1.564198e+00      5.858535e+00     4.731591e+00

  CustomerID  Recency_scaled  Frequency_scaled  Monetary_scaled
0      12346        1.461993         -0.955214         3.706225
1      12347       -2.038734          1.074425         1.411843
2      12348        0.373104          0.386304         0.716489
3      12349       -0.623086         -0.955214         0.698739
4      12350        1.424558         -0.955214        -0.618962


### 4.5  CORRELATION HEATMAP

In [131]:
plt.figure(figsize=(6, 5))
corr = rfm[['Recency', 'Frequency', 'Monetary']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('RFM Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('rfm_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Save the Artifacts

In [132]:
rfm.to_csv('rfm_table.csv', index=False)
np.save('rfm_scaled.npy', rfm_scaled_array)


# 5.0 KMeans Clustering

In [133]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
sns.set_style('whitegrid')

In [134]:
rfm = pd.read_csv('rfm_table.csv')
rfm_scaled_array = np.load('rfm_scaled.npy')
print(f"Clustering {rfm_scaled_array.shape[0]:,} customers "
      f"on {rfm_scaled_array.shape[1]} scaled RFM features")


Clustering 4,338 customers on 3 scaled RFM features


## 5.1  FIND THE RIGHT NUMBER OF CLUSTERS (k)
###We test k from 2 to 10 and track TWO metrics:





*   **INERTIA** (Elbow Method)
   Sum of squared distances of points to their cluster center.
   Always decreases as k increases (more clusters = tighter
   fit). We look for the "elbow" — the point where adding
   more clusters stops giving a big improvement.
*  **SILHOUETTE SCORE** Measures how well-separated clusters are, from -1 to 1. Higher = clusters are dense and well separated from eachother. This is more reliable than inertia alone becauseinertia keeps dropping even when extra clusters don't add real business meaning.


  

In [135]:
inertia, sil_scores = [], []
K = range(2, 11)

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled_array)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled_array, labels))
    print(f"k={k}: inertia={km.inertia_:8.1f}   silhouette={sil_scores[-1]:.4f}")

k=2: inertia=  6481.2   silhouette=0.4329
k=3: inertia=  4867.8   silhouette=0.3365
k=4: inertia=  3938.5   silhouette=0.3371
k=5: inertia=  3296.0   silhouette=0.3161
k=6: inertia=  2855.0   silhouette=0.3133
k=7: inertia=  2548.9   silhouette=0.3100
k=8: inertia=  2336.8   silhouette=0.3008
k=9: inertia=  2155.6   silhouette=0.2817
k=10: inertia=  1999.9   silhouette=0.2787


###  5.2  Plot both metrics side by side

In [136]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(K, inertia, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method (Inertia)', fontweight='bold')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (lower = tighter clusters)')
axes[0].axvline(4, color='red', linestyle='--', alpha=0.6, label='Chosen k=4')
axes[0].legend()

axes[1].plot(K, sil_scores, marker='o', color='seagreen', linewidth=2)
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score (higher = better separation)')
axes[1].axvline(4, color='red', linestyle='--', alpha=0.6, label='Chosen k=4')
axes[1].legend()

plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

### **WHY k=4**:
#### - Silhouette peaks at k=2 (0.43) and k=4 (0.34) — almost tied   with k=3 (0.34).


### - The elbow plot visibly flattens out around k=4-5 — beyond  that, extra clusters buy very little reduction in inertia.
### - k=4 gives FOUR distinct, business-interpretable segments, which is the sweet spot between simplicity   and granularity for a marketing team to act on.

### 5.3  FIT THE FINAL KMEANS MODEL

In [137]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled_array)

final_silhouette = silhouette_score(rfm_scaled_array, rfm['Cluster'])
print(f"\n Final model: k=4, Silhouette Score = {final_silhouette:.4f}")


 Final model: k=4, Silhouette Score = 0.3371



### 5.4  INTERPRET EACH CLUSTER

In [138]:
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
cluster_summary['Customer Count'] = rfm['Cluster'].value_counts().sort_index()
print("\n Cluster profiles (mean values):")
print(cluster_summary)


 Cluster profiles (mean values):
         Recency  Frequency  Monetary  Customer Count
Cluster                                              
0           18.1        2.1     551.8             837
1           12.1       13.7    8074.3             716
2           71.1        4.1    1802.8            1173
3          182.5        1.3     343.5            1612


###  5.5  Visualize cluster profile

In [139]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['Recency', 'Frequency', 'Monetary'],
                            ['steelblue', 'seagreen', 'coral']):
    sns.barplot(x=cluster_summary.index, y=cluster_summary[col], ax=ax,
                hue=cluster_summary.index, palette=[color]*4, legend=False)
    ax.set_title(f'Mean {col} by Cluster')
    ax.set_xlabel('Cluster')
plt.suptitle('Step 5.5 — Cluster Profiles', fontweight='bold')
plt.tight_layout()
plt.savefig('kmeans_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6  LABEL EACH CLUSTER WITH A BUSINESS-FRIENDLY NAME

In [140]:
cluster_summary['RFM_Score'] = (
    cluster_summary['Frequency'].rank() +
    cluster_summary['Monetary'].rank() -
    cluster_summary['Recency'].rank()
)
ranked_clusters = cluster_summary.sort_values('RFM_Score', ascending=False).index.tolist()

label_names = ['High-Value', 'Regular', 'Occasional', 'At-Risk']
cluster_to_label = dict(zip(ranked_clusters, label_names))

print("\n  Cluster → Label mapping (based on RFM ranking):")
for cluster_id, label in cluster_to_label.items():
    print(f"   Cluster {cluster_id} → {label}")

rfm['Segment'] = rfm['Cluster'].map(cluster_to_label)

print("\n Final labeled RFM table (sample):")
print(rfm.head(10))


  Cluster → Label mapping (based on RFM ranking):
   Cluster 1 → High-Value
   Cluster 2 → Regular
   Cluster 0 → Occasional
   Cluster 3 → At-Risk

 Final labeled RFM table (sample):
   CustomerID  Recency  Frequency  Monetary  Recency_log  Frequency_log  \
0       12346      326          1  77183.60     5.789960       0.693147   
1       12347        2          7   4310.00     1.098612       2.079442   
2       12348       75          4   1797.24     4.330733       1.609438   
3       12349       19          1   1757.55     2.995732       0.693147   
4       12350      310          1    334.40     5.739793       0.693147   
5       12352       36          8   2506.04     3.610918       2.197225   
6       12353      204          1     89.00     5.323010       0.693147   
7       12354      232          1   1079.40     5.451038       0.693147   
8       12355      214          1    459.40     5.370638       0.693147   
9       12356       23          3   2811.43     3.178054       1.

### **WHY** BUSINESS-FRIENDLY NAME

* A marketing team doesn't think in "Cluster 0, 1, 2, 3" — they think in "High-Value", "At-Risk".
* We assign labels by RANKING clusters on Monetary + Frequency(higher = better) and Recency (lower = better), rather than
 hardcoding cluster numbers — because KMeans cluster numbers
 are arbitrary and can change between runs/random seeds.




###  5.7  Segment-level summary with labels

In [141]:
segment_summary = rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
segment_summary['Customer Count'] = rfm['Segment'].value_counts()
segment_summary['% of Customers'] = (segment_summary['Customer Count'] / len(rfm) * 100).round(1)
segment_summary = segment_summary.reindex(label_names)  # nice order
print("\n Final segment summary:")
print(segment_summary)


 Final segment summary:
            Recency  Frequency  Monetary  Customer Count  % of Customers
Segment                                                                 
High-Value     12.1       13.7    8074.3             716            16.5
Regular        71.1        4.1    1802.8            1173            27.0
Occasional     18.1        2.1     551.8             837            19.3
At-Risk       182.5        1.3     343.5            1612            37.2


###  5.8  Visualize segment sizes

In [142]:
plt.figure(figsize=(8, 6))
colors_map = {'High-Value':'#2ca02c', 'Regular':'#1f77b4',
              'Occasional':'#ff7f0e', 'At-Risk':'#d62728'}
seg_counts = rfm['Segment'].value_counts().reindex(label_names)
plt.pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
        colors=[colors_map[s] for s in seg_counts.index],
        wedgeprops={'edgecolor':'white','linewidth':1.5}, startangle=90)
plt.title('Customer Segment Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('kmeans_segment_pie.png', dpi=150, bbox_inches='tight')
plt.show()
rfm.to_csv('rfm_clustered.csv', index=False)

#6.0 **Product Recommandation System**

In [143]:
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv('online_retail_clean.csv')
df['CustomerID'] = df['CustomerID'].astype(str)

### 6.1  THE CORE IDEA — item-based collaborative filtering

 "Customers who bought X also bought Y" — that's the entire
 concept. We don't need to know ANYTHING about what the products actually are (no descriptions, no categories). We only look at customer purchase PATTERNS.


*   Step 1: Build a Customer × Product matrix — each cell says
         how many of that product a customer bought in total.

*   Step 2: Compare PRODUCTS to each other (not customers) by
         looking at the column vectors of that matrix.
         Two products are "similar" if they tend to be bought
         by the same customers in similar quantities.

*  Step 3: Use COSINE SIMILARITY to measure that closeness.











### 6.2  FILTER OUT VERY RARE PRODUCTS

In [144]:
product_customer_counts = df.groupby('Description')['CustomerID'].nunique()
valid_products = product_customer_counts[product_customer_counts >= 5].index

print(f"Total products          : {df['Description'].nunique():,}")
print(f"Products kept (≥5 buyers): {len(valid_products):,}")
print(f"Products dropped (rare)  : {df['Description'].nunique() - len(valid_products):,}")

df_filtered = df[df['Description'].isin(valid_products)]

Total products          : 3,877
Products kept (≥5 buyers): 3,260
Products dropped (rare)  : 617


### 6.3  BUILD THE CUSTOMER × PRODUCT PIVOT TABLE

In [145]:
pivot = df_filtered.pivot_table(
    index='CustomerID',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)

print(f"\nPivot table shape: {pivot.shape[0]:,} customers × {pivot.shape[1]:,} products")
print(f"Matrix size: {pivot.shape[0] * pivot.shape[1]:,} cells "
      f"(mostly zeros — this is a 'sparse' matrix)")


Pivot table shape: 4,338 customers × 3,260 products
Matrix size: 14,141,880 cells (mostly zeros — this is a 'sparse' matrix)


### 6.4  COMPUTE ITEM-ITEM COSINE SIMILARITY

Cosine similarity measures the angle between two vectors,
 ignoring their magnitude. For two products, their "vector"
 is the column of how much each customer bought of them.

In [146]:
item_similarity_matrix = cosine_similarity(pivot.T)

# Wrap in a DataFrame so we can look up by product NAME
item_sim_df = pd.DataFrame(
    item_similarity_matrix,
    index=pivot.columns,
    columns=pivot.columns
)

print(f"\nItem similarity matrix shape: {item_sim_df.shape}")
print("(every product's similarity score to every other product)")


Item similarity matrix shape: (3260, 3260)
(every product's similarity score to every other product)


###6.5  BUILD THE RECOMMENDATION FUNCTION

In [147]:
def get_recommendations(product_name, top_n=5):
    """
    Given a product name, return the top_n most similar products
    based on customer co-purchase patterns.
    """
    if product_name not in item_sim_df.index:
        return f" Product '{product_name}' not found in catalog."

    # Get similarity scores for this product against all others,
    # drop itself (a product is always 100% similar to itself),
    # then sort descending and take the top N.
    similar_scores = (
        item_sim_df[product_name]
        .drop(product_name)
        .sort_values(ascending=False)
        .head(top_n)
    )
    return similar_scores


###  6.6  Test the recommender on a few example products

In [148]:
test_products = [
    'WHITE HANGING HEART T-LIGHT HOLDER',
    'JUMBO BAG RED RETROSPOT',
    'RABBIT NIGHT LIGHT'
]

for product in test_products:
    print(f"\n Top 5 recommendations for: '{product}'")
    recs = get_recommendations(product, top_n=5)
    if isinstance(recs, str):
        print(recs)
    else:
        for i, (name, score) in enumerate(recs.items(), 1):
            print(f"   {i}. {name}  (similarity: {score:.3f})")


 Top 5 recommendations for: 'WHITE HANGING HEART T-LIGHT HOLDER'
   1. GIN + TONIC DIET METAL SIGN  (similarity: 0.750)
   2. RED HANGING HEART T-LIGHT HOLDER  (similarity: 0.659)
   3. WASHROOM METAL SIGN  (similarity: 0.644)
   4. LAUNDRY 15C METAL SIGN  (similarity: 0.642)
   5. GREEN VINTAGE SPOT BEAKER  (similarity: 0.631)

 Top 5 recommendations for: 'JUMBO BAG RED RETROSPOT'
   1. JUMBO BAG STRAWBERRY  (similarity: 0.902)
   2. JUMBO BAG PINK POLKADOT  (similarity: 0.897)
   3. JUMBO BAG OWLS  (similarity: 0.801)
   4. JUMBO BAG PINK VINTAGE PAISLEY  (similarity: 0.789)
   5. JUMBO BAG APPLES  (similarity: 0.758)

 Top 5 recommendations for: 'RABBIT NIGHT LIGHT'
   1. PLASTERS IN TIN SPACEBOY  (similarity: 0.777)
   2.  DOLLY GIRL BEAKER  (similarity: 0.776)
   3. ROUND SNACK BOXES SET OF4 WOODLAND   (similarity: 0.774)
   4. CARD DOLLY GIRL   (similarity: 0.772)
   5. SPACEBOY BEAKER  (similarity: 0.771)


### 6.7 Visualize

In [149]:
sample_product = test_products[0]
sample_recs = get_recommendations(sample_product, top_n=5)

plt.figure(figsize=(9, 5))
sample_recs.sort_values().plot(kind='barh', color='mediumseagreen')
plt.title(f"Top 5 Similar Products to:\n'{sample_product}'", fontweight='bold')
plt.xlabel('Cosine Similarity Score')
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig('recommendation_example.png', dpi=150, bbox_inches='tight')
plt.show()

###6.8  SANITY CHECK — similarity score distribution

In [150]:
sample_scores = item_similarity_matrix[np.triu_indices_from(item_similarity_matrix, k=1)]
sample_scores = np.random.choice(sample_scores, size=50000, replace=False)  # sample for speed

plt.figure(figsize=(9, 5))
plt.hist(sample_scores, bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of Pairwise Product Similarity Scores', fontweight='bold')
plt.xlabel('Cosine Similarity')
plt.ylabel('Number of Product Pairs')
plt.yscale('log')
plt.tight_layout()
plt.savefig('similarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

###6.9  SAVE ARTIFACTS FOR THE STREAMLIT APP

In [151]:
import joblib
joblib.dump(item_sim_df, 'item_similarity_matrix.pkl')

['item_similarity_matrix.pkl']